In [1]:
# generate_cosmo_files.py
# Env: needs rdkit, opi (ORCA python interface), xtb (on PATH), ORCA on PATH
# Run inside your OPI venv

from pathlib import Path
import re
import shutil
import subprocess
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

from opi.core import Calculator
from opi.input.structures.structure import Structure

# ---- config ----
GRID_CSV = "first_100.csv"        # columns: Solvent, Smiles, Temperature
CLOZAPINE_SMILES = "ClC1=CC=C2NC=3C=CC=CC3C(=NC2=C1)N4CCN(C)CC4"  # exact SMILES from Clozapine_dataset.csv
WORKDIR = Path("cosmo_calcs")
NCORES = 4
WORKDIR.mkdir(exist_ok=True)

def safe_name(name: str) -> str:
    """Turn a solvent name into a filesystem-safe folder/basename."""
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(name)).strip("_")

def smiles_to_xyz_block(smiles: str, n_confs: int = 5, seed: int = 42) -> str:
    """Rough starting geometry only — this is refined by xtb afterward,
    so a fast RDKit/MMFF embedding is sufficient here."""
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    cids = AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, randomSeed=seed)
    energies = []
    for cid in cids:
        ff = AllChem.MMFFGetMoleculeForceField(mol, AllChem.MMFFGetMoleculeProperties(mol), confId=cid)
        if ff is None:
            ff = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
        ff.Minimize(maxIts=2000)
        energies.append((ff.CalcEnergy(), cid))
    best_cid = min(energies, key=lambda t: t[0])[1]
    return Chem.MolToXYZBlock(mol, confId=best_cid)

def xtb_optimize_xyz(xyz_path: Path, charge: int = 0, mult: int = 1) -> Path:
    """Refine an RDKit-generated geometry using GFN2-xTB optimization,
    matching the literature methodology. Returns path to xtbopt.xyz."""
    if shutil.which("xtb") is None:
        raise RuntimeError("xtb binary not found on PATH — install it (e.g. `conda install -c conda-forge xtb`)")

    work_dir = xyz_path.parent
    cmd = ["xtb", xyz_path.name, "--opt", "--gfn", "2", "--chrg", str(charge), "--uhf", str(mult - 1)]

    result = subprocess.run(cmd, cwd=work_dir, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"xtb optimization failed:\n{result.stdout}\n{result.stderr}")

    opt_xyz = work_dir / "xtbopt.xyz"
    if not opt_xyz.exists():
        raise RuntimeError("xtb finished but xtbopt.xyz was not produced")

    return opt_xyz

def setup_calc(basename: str, working_dir: Path, structure: Structure, ncores: int = NCORES) -> Calculator:
    calc = Calculator(basename=basename, working_dir=working_dir)
    calc.structure = structure
    # solvent name in COSMORS(...) only affects an auxiliary reference file we discard;
    # what we keep is <basename>.solute.orcacosmo, which is this molecule's own sigma profile
    calc.input.add_arbitrary_string("!COSMORS(water)")
    calc.input.ncores = ncores
    return calc

def run_and_get_cosmo(smiles: str, name: str) -> Path | None:
    mol_dir = WORKDIR / name
    mol_dir.mkdir(exist_ok=True)
    out_file = mol_dir / f"{name}.solute.orcacosmo"
    if out_file.exists():
        return out_file  # resume support: skip already-done molecules

    # 1) rough RDKit/MMFF starting geometry
    try:
        xyz = smiles_to_xyz_block(smiles)
    except Exception as e:
        print(f"[{name}] RDKit embedding failed: {e}")
        return None

    xyz_path = mol_dir / f"{name}_rdkit.xyz"
    xyz_path.write_text(xyz)

    # 2) refine with GFN2-xTB, matching the literature methodology
    try:
        opt_xyz_path = xtb_optimize_xyz(xyz_path)
    except Exception as e:
        print(f"[{name}] xtb optimization failed: {e}")
        return None

    structure = Structure.from_xyz(opt_xyz_path)

    # 3) ORCA COSMO-RS single point on the xtb-optimized geometry
    calc = setup_calc(basename=name, working_dir=mol_dir, structure=structure)
    calc.write_input()
    try:
        calc.run()
    except Exception as e:
        print(f"[{name}] ORCA run failed: {e}")
        return None

    output = calc.get_output()
    if not output.terminated_normally():
        print(f"[{name}] ORCA did not terminate normally")
        return None

    return out_file if out_file.exists() else None

def main():
    grid_df = pd.read_csv(GRID_CSV)
    unique_solvents = grid_df.drop_duplicates(subset="Smiles")[["Solvent", "Smiles"]].reset_index(drop=True)
    print(f"{len(grid_df)} grid rows -> {len(unique_solvents)} unique solvents to run through ORCA")

    log_rows = []

    # --- solute: clozapine, done once ---
    cosmo_path = run_and_get_cosmo(CLOZAPINE_SMILES, "clozapine")
    log_rows.append({"name": "clozapine", "smiles": CLOZAPINE_SMILES, "cosmo_file": str(cosmo_path)})

    # --- solvents ---
    for i, row in unique_solvents.iterrows():
        smi = row["Smiles"]
        name = safe_name(row["Solvent"])
        print(f"[{i+1}/{len(unique_solvents)}] {name}: {smi}")
        cosmo_path = run_and_get_cosmo(smi, name)
        log_rows.append({"name": name, "solvent": row["Solvent"], "smiles": smi, "cosmo_file": str(cosmo_path)})

    pd.DataFrame(log_rows).to_csv(WORKDIR / "cosmo_manifest.csv", index=False)
    print("Done. Manifest written to", WORKDIR / "cosmo_manifest.csv")

if __name__ == "__main__":
    main()

1200 grid rows -> 100 unique solvents to run through ORCA
[1/100] 1-Thiapropyl_-Benzene: C1=CC=CC=C1SCC
[2/100] E_-Citral: CC(=CCC/C(=C/C=O)/C)C
[3/100] Z_-3-Hexen-1-Yl_Formate: CC\C=C/CCOC=O
[4/100] Z_-3-Hexen-1-ol: CC\C=C/CCO
[5/100] Z_-4-Hepten-1-ol: CC\C=C/CCCO
[6/100] Z_-6-Nonen-1-Al: CC\C=C/CCCCC=O
[7/100] Z_-Verbenol: CC1=C[C@H]([C@@H]2C[C@H]1C2(C)C)O
[8/100] 1_1-Dimethylcyclohexane: CC1(C)CCCCC1
[9/100] 1_2_3-Trimethylbenzene: CC(C(C)=CC=C1)=C1C
[10/100] 1_2-Cyclohexanedione: C1CCC(=O)C(=O)C1
[11/100] 1_3-Butanediol: CC(O)CCO
[12/100] 1_3-Cyclohexadiene: C1=CCCC=C1
[13/100] 1_3-Dioxolane: C1OCCO1
[14/100] 1_4-Thioxane: C1CSCCO1
[15/100] 1_8-Cineole_Eucalyptol: CC1(C2CCC(O1)(CC2)C)C
[16/100] 1-Hepten-3-ol: CCCCC(C=C)O
[17/100] 1-Nonanal: CCCCCCCCC=O
[18/100] 1-Nonyne: C#CCCCCCCC
[19/100] 1-Octanal: CCCCCCCC=O
[20/100] 1-Octanol: CCCCCCCCO
[21/100] 1-Penten-3-ol: CCC(C=C)O
[22/100] 1-Phenyl-3-Methylbutane: CC(C)CCC1=CC=CC=C1
[23/100] 1-Propene_3_3_-Thiobis-: C=CCSCC=C
[24/100] 2_

In [1]:
# compute_solubility_grid_fsolve.py
#
# Batch version of the validated paracetamol test methodology: same solver
# (non-iterative closed-form + fsolve-based iterative), applied across the
# whole solvent grid using a pre-built COSMO manifest instead of generating
# COSMO files on the fly.

import numpy as np
import pandas as pd
from scipy.optimize import fsolve

from opencosmorspy.parameterization import openCOSMORS24a
from opencosmorspy.cosmors import COSMORS

# ---- config ----
GRID_CSV = "first_100.csv"    # columns: Solvent, Smiles, MWt (g/mol), Temperature, LogS (empty)
MANIFEST_CSV = "cosmo_calcs/cosmo_manifest.csv"
OUT_CSV = "clozapine_pure_solvent_pretrain_100.csv"

R = 8.314
DELTA_H_FUSION = 34350.0
T_FUSION = 456.43
MW_CLOZAPINE = 326.82

def compute_ln_gamma(crs: COSMORS, mole_fractions: list[float], temp: float) -> float:
    """Compute ln(gamma) for a given composition at specified temperature using openCOSMO-RS"""
    crs.clear_jobs()
    crs.add_job(np.array(mole_fractions), temp, refst="pure_component")
    return crs.calculate()["tot"]["lng"][0][0]

def solubility(crs: COSMORS, delta_h_fus: float, t_fus: float, temp: float, iterative: bool) -> float:
    """Estimate solubility from openCOSMO-RS ln(gamma).
    iterative=False -> closed-form non-iterative estimate (infinite dilution gamma)
    iterative=True  -> fsolve on the full equilibrium condition
    """
    rhs = -delta_h_fus / R * (1 / temp - 1 / t_fus)

    if not iterative:
        ln_gamma_inf = compute_ln_gamma(crs, [0.0, 1.0], temp)
        return float(np.exp(rhs - ln_gamma_inf))

    def equation(x_guess) -> float:
        """Equilibrium condition to be solved"""
        x_val = float(np.atleast_1d(x_guess)[0])
        x_val = max(1e-15, min(1.0, x_val))
        x = np.array([x_val, 1 - x_val])
        ln_gamma = compute_ln_gamma(crs, x, temp)
        return np.abs(ln_gamma + np.log(x_val) - rhs)

    result = fsolve(equation, 1e-5)

    x_sol = float(np.atleast_1d(result)[0])
    x_sol = max(1e-15, min(1.0, x_sol))
    return x_sol

def mole_frac_to_logS(x_solute: float, mw_solute: float, mw_solvent: float) -> float:
    mass_solute = x_solute * mw_solute
    mass_solvent = (1 - x_solute) * mw_solvent
    w = mass_solute / (mass_solute + mass_solvent)
    return np.log10(100 * w)

def main():
    grid_df = pd.read_csv(GRID_CSV)
    manifest = pd.read_csv(MANIFEST_CSV)

    clozapine_row = manifest[manifest["name"] == "clozapine"].iloc[0]
    solute_path = clozapine_row["cosmo_file"]

    solvent_manifest = manifest[manifest["name"] != "clozapine"][["smiles", "cosmo_file"]]
    merged = grid_df.merge(solvent_manifest, left_on="Smiles", right_on="smiles", how="left")

    missing_cosmo = merged["cosmo_file"].isna().sum()
    if missing_cosmo:
        print(f"[warning] {missing_cosmo} rows have no matching cosmo_file — these will be skipped")

    crs = COSMORS(par=openCOSMORS24a())
    results = []
    current_solvent_smi = None

    merged = merged.sort_values("Solvent").reset_index(drop=True)

    for i, row in merged.iterrows():
        smi = row["Smiles"]
        T = row["Temperature"]
        cosmo_file = row["cosmo_file"]
        mw_solvent = row["MWt (g/mol)"]

        if pd.isna(cosmo_file) or pd.isna(mw_solvent):
            continue

        if smi != current_solvent_smi:
            crs.clear_molecules()
            crs.add_molecule([solute_path])
            crs.add_molecule([cosmo_file])
            current_solvent_smi = smi

        try:
            x_non_iter = solubility(crs, DELTA_H_FUSION, T_FUSION, T, iterative=False)
            x_iter = solubility(crs, DELTA_H_FUSION, T_FUSION, T, iterative=True)
            logS_non_iter = mole_frac_to_logS(x_non_iter, MW_CLOZAPINE, float(mw_solvent))
            logS_iter = mole_frac_to_logS(x_iter, MW_CLOZAPINE, float(mw_solvent))
        except Exception as e:
            print(f"[{row['Solvent']}, {T}K] solve failed: {e}")
            continue

        results.append({
            "Solvent": row["Solvent"],
            "SMILES_1": smi,
            "comp_1": 100.0,
            "Temperature": T,
            "MWt (g/mol)": mw_solvent,
            "x_solute_non_iter": x_non_iter,
            "x_solute_iter": x_iter,
            "LogS_non_iter": logS_non_iter,
            "LogS_iter": logS_iter,
        })

        if (i + 1) % 50 == 0:
            print(f"[{i+1}/{len(merged)}] rows processed")

    out_df = pd.DataFrame(results)
    out_df.to_csv(OUT_CSV, index=False)
    print(f"Wrote {len(out_df)} rows to {OUT_CSV}")

if __name__ == "__main__":
    main()

[50/1200] rows processed
[100/1200] rows processed
[150/1200] rows processed
[200/1200] rows processed
[250/1200] rows processed
[300/1200] rows processed
[350/1200] rows processed
[400/1200] rows processed
[450/1200] rows processed
[500/1200] rows processed
[550/1200] rows processed
[600/1200] rows processed
[650/1200] rows processed
[700/1200] rows processed
[750/1200] rows processed
[800/1200] rows processed
[850/1200] rows processed
[900/1200] rows processed
[950/1200] rows processed
[1000/1200] rows processed
[1050/1200] rows processed
[1100/1200] rows processed
[1150/1200] rows processed
[1200/1200] rows processed
Wrote 1200 rows to clozapine_pure_solvent_pretrain_100.csv
